# <font color="#418FDE" size="6.5" uppercase>**Cluster und PCA**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Implementieren K-Means-Schritte mit Distanzen, Zuordnung und Zentrenaktualisierung. 
- Berechnen Hauptkomponenten, erklärte Varianz und Projektionen mit NumPy. 
- Analysieren Clusterqualität, Gegenbeispiele und einfache Anomaliehinweise. 


## **1. K Means manuell**

### **1.1. Distanzen berechnen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_01_01.jpg?v=1784793839" width="250">



>* Distanzen messen Ähnlichkeit zwischen Datenpunkten
>* Nächste Zentren bestimmen spätere Clusterzuordnung

>* Euklidische Distanz braucht vergleichbare Merkmalsgrößen
>* Skalierung verhindert verzerrte Näheentscheidungen

>* Distanzen vergleichen Punkte mit allen Zentren
>* Iterationen verändern Zuordnung und Cluster



In [ ]:
#@title Python-Code - Distanzen berechnen

# Wir berechnen Distanzen für K-Means manuell.
# Euklidische Abstände entscheiden die nächste Zuordnung.
# Die Ausgabe zeigt Punkte, Zentren und Abstände.

import numpy as np
import matplotlib.pyplot as plt

# Kleine zweidimensionale Daten machen die Rechnung übersichtlich.
points = np.array([[1.0, 2.0], [2.0, 1.0], [5.0, 4.0], [6.0, 5.0]])
centers = np.array([[1.0, 1.0], [6.0, 4.0]])

# Diese Prüfung verhindert unpassende Punkt- und Zentrumformen.
if points.shape[1] != centers.shape[1]:
    raise ValueError("Punkte und Zentren brauchen gleich viele Merkmale.")

# Broadcasting bildet alle Punkt-Zentrum-Differenzen gleichzeitig.
differences = points[:, np.newaxis, :] - centers[np.newaxis, :, :]
squared_differences = differences ** 2

# Die euklidische Distanz ist die Wurzel der Quadratsumme.
squared_distances = np.sum(squared_differences, axis=2)
distances = np.sqrt(squared_distances)

# Das kleinste Distanzfeld liefert die vorläufige Clusterzuordnung.
assignments = np.argmin(distances, axis=1)
rounded_distances = np.round(distances, 2)

print("Distanzen zu Zentrum 0 und 1:")
for index, row in enumerate(rounded_distances):
    print(f"Punkt {index}: {row[0]:.2f}, {row[1]:.2f} -> Cluster {assignments[index]}")

# Die Grafik verbindet jeden Punkt mit seinem nächsten Zentrum.
fig, ax = plt.subplots(figsize=(6, 4))
colors = np.array(["tab:blue", "tab:orange"])

ax.scatter(points[:, 0], points[:, 1], c=colors[assignments], s=80, label="Punkte")
ax.scatter(centers[:, 0], centers[:, 1], c=colors, marker="X", s=180, label="Zentren")

for point, cluster in zip(points, assignments):
    center = centers[cluster]
    ax.plot([point[0], center[0]], [point[1], center[1]], color=colors[cluster])

ax.set_title("K-Means: Distanzen zu aktuellen Zentren")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **1.2. Zentren und Zuordnung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_01_02.jpg?v=1784793841" width="250">



>* Datenpunkte wandern zum nächstgelegenen Zentrum
>* Zuordnungen ändern sich mit neuen Zentren

>* Zentren wandern zum Durchschnitt ihrer Punkte
>* Neue Zentren stabilisieren die Cluster schrittweise

>* Zentrenverschiebungen können Clusterwechsel auslösen
>* Wiederholung erzeugt konsistente, ähnliche Gruppen



In [ ]:
#@title Python-Code - Zentren und Zuordnung

# Wir ordnen Punkte dem nächsten Zentrum zu.
# Danach berechnen wir neue Clusterzentren manuell.
# Die Grafik zeigt Zuordnung und Verschiebung.

import numpy as np
import matplotlib.pyplot as plt

# Kleine zweidimensionale Datenpunkte bleiben gut nachvollziehbar.
points = np.array([[1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0], [6.0, 8.0], [8.0, 8.0]])

# Zwei Startzentren werden bewusst einfach gewählt.
centers = np.array([[1.0, 1.0], [8.0, 8.0]])

# Die Formprüfung verhindert unpassende Punkt-Zentrum-Kombinationen.
if points.shape[1] != centers.shape[1]:
    raise ValueError("Punkte und Zentren brauchen gleich viele Merkmale.")

# Jede Zeile enthält die Abstände eines Punktes zu allen Zentren.
differences = points[:, np.newaxis, :] - centers[np.newaxis, :, :]
distances = np.sqrt(np.sum(differences ** 2, axis=2))

# Das kleinste Distanzfeld bestimmt die Clusterzuordnung.
labels = np.argmin(distances, axis=1)

# Für jedes Cluster wird der Durchschnitt seiner Punkte berechnet.
new_centers = centers.copy()
for cluster_id in range(len(centers)):
    cluster_points = points[labels == cluster_id]
    if len(cluster_points) > 0:
        new_centers[cluster_id] = cluster_points.mean(axis=0)

# Kurze Ausgaben zeigen die wichtigsten Rechenschritte.
print("Abstände Punkt 1 zu Zentren:", np.round(distances[0], 2))
print("Zuordnung aller Punkte:", labels.tolist())
print("Alte Zentren:", np.round(centers, 2).tolist())
print("Neue Zentren:", np.round(new_centers, 2).tolist())

# Die Grafik verbindet Zuordnung und Zentrenverschiebung sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(points[:, 0], points[:, 1], c=labels, s=80, cmap="viridis")
ax.scatter(centers[:, 0], centers[:, 1], marker="x", s=160, c="red", label="alte Zentren")
ax.scatter(new_centers[:, 0], new_centers[:, 1], marker="P", s=160, c="black", label="neue Zentren")

# Pfeile zeigen, wohin sich die Zentren bewegen.
for old_center, new_center in zip(centers, new_centers):
    movement = new_center - old_center
    ax.arrow(old_center[0], old_center[1], movement[0], movement[1], head_width=0.18, length_includes_head=True, color="gray")

ax.set_title("K-Means: Zuordnung und neue Zentren")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **1.3. K Means Iterationen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_01_03.jpg?v=1784793842" width="250">



>* Punkte zuordnen, Zentren neu berechnen
>* Wiederholung stabilisiert Cluster und Zentren

>* Punkte wechseln Cluster durch verschobene Zentren
>* Distanzen, Zuordnung und Zentren bauen aufeinander auf

>* Konvergenz garantiert nicht die beste Lösung
>* Mehrere Starts testen, Cluster fachlich prüfen



In [ ]:
#@title Python-Code - K Means Iterationen

# Dieses Beispiel zeigt K Means Iterationen manuell.
# Distanzen bestimmen Zuordnungen und neue Zentren.
# Die Ausgabe zeigt schrittweise stabilere Cluster.

import numpy as np
import matplotlib.pyplot as plt

# Kleine zweidimensionale Datenpunkte bleiben gut nachvollziehbar.
points = np.array([
    [1.0, 1.0], [1.5, 2.0], [2.0, 1.5],
    [6.0, 5.0], [7.0, 5.5], [6.5, 6.5],
])

# Zwei Startzentren werden bewusst nicht perfekt gewählt.
centers = np.array([[1.0, 1.0], [7.0, 5.5]])
previous_labels = np.full(points.shape[0], -1)
center_history = [centers.copy()]

# Jede Runde berechnet Distanzen, Zuordnungen und Mittelwerte.
for iteration in range(1, 6):
    distances = np.linalg.norm(points[:, None, :] - centers[None, :, :], axis=2)
    labels = np.argmin(distances, axis=1)

    new_centers = centers.copy()
    for cluster_id in range(2):
        cluster_points = points[labels == cluster_id]
        if len(cluster_points) > 0:
            new_centers[cluster_id] = cluster_points.mean(axis=0)

    moved = np.linalg.norm(new_centers - centers, axis=1)
    changed = int(np.sum(labels != previous_labels))
    print(f"Iteration {iteration}: Wechsel={changed}, Bewegung={np.round(moved, 2)}")

    centers = new_centers
    center_history.append(centers.copy())
    if np.array_equal(labels, previous_labels):
        break
    previous_labels = labels.copy()

# Eine einfache Prüfung schützt vor unerwarteten Formen.
if points.shape[1] != 2 or centers.shape != (2, 2):
    raise ValueError("Dieses Beispiel erwartet zweidimensionale Punkte.")

print(f"Endzentren: {np.round(centers, 2).tolist()}")
print(f"Endzuordnung: {labels.tolist()}")

# Die Grafik zeigt Punkte und den Weg der Zentren.
fig, ax = plt.subplots(figsize=(6, 4))
colors = np.array(["tab:blue", "tab:orange"])
ax.scatter(points[:, 0], points[:, 1], c=colors[labels], s=80, label="Datenpunkte")

history = np.array(center_history)
for cluster_id in range(2):
    ax.plot(history[:, cluster_id, 0], history[:, cluster_id, 1], marker="x")

ax.scatter(centers[:, 0], centers[:, 1], c="black", s=120, marker="*", label="Endzentren")
ax.set_title("Manuelle K Means Iterationen")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



## **2. Clusterqualität bewerten**

### **2.1. Inertia verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_02_01.jpg?v=1784793850" width="250">



>* Niedrige Inertia zeigt kompakte Cluster.
>* Merkmalsskalierung beeinflusst Distanzen stark.

>* Mehr Cluster senken Inertia meist automatisch
>* Interpretierbare Gruppen bleiben wichtiger als Minimalwerte

>* Inertia bevorzugt kompakte, runde Cluster
>* Immer mit Visualisierung und Fachwissen prüfen



In [ ]:
#@title Python-Code - Inertia verstehen

# Dieses Beispiel macht Inertia rechnerisch sichtbar.
# Wir vergleichen kompakte und breite Cluster.
# Die Ausgabe zeigt quadrierte Abstände verständlich.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import sklearn

# Zwei kleine Datensätze zeigen unterschiedliche Kompaktheit.
compact_points = np.array([
    [0.0, 0.1], [0.2, -0.1], [-0.1, 0.0], [0.1, 0.2],
    [4.0, 4.1], [4.2, 3.9], [3.9, 4.0], [4.1, 4.2],
])

wide_points = np.array([
    [-0.8, 0.7], [0.9, -0.6], [-0.7, -0.8], [0.8, 0.9],
    [3.0, 5.0], [5.1, 3.1], [2.9, 3.0], [5.0, 5.2],
])

# K-Means findet für beide Datensätze zwei Zentren.
compact_model = KMeans(n_clusters=2, n_init=10, random_state=42)
wide_model = KMeans(n_clusters=2, n_init=10, random_state=42)
compact_labels = compact_model.fit_predict(compact_points)
wide_labels = wide_model.fit_predict(wide_points)

# Inertia ist die Summe quadrierter Abstände zum eigenen Zentrum.
compact_distances = compact_points - compact_model.cluster_centers_[compact_labels]
wide_distances = wide_points - wide_model.cluster_centers_[wide_labels]
compact_manual = np.sum(compact_distances ** 2)
wide_manual = np.sum(wide_distances ** 2)

# Eine einfache Prüfung schützt vor Formfehlern.
if compact_points.shape != wide_points.shape:
    raise ValueError("Beide Datensätze brauchen dieselbe Form.")

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Kompakte Cluster, Inertia: {compact_model.inertia_:.2f}")
print(f"Breite Cluster, Inertia: {wide_model.inertia_:.2f}")
print(f"Manuell nachgerechnet, kompakt: {compact_manual:.2f}")
print(f"Manuell nachgerechnet, breit: {wide_manual:.2f}")

# Die Grafik zeigt, warum breite Cluster höhere Inertia haben.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(compact_points[:, 0], compact_points[:, 1], label="kompakt")
ax.scatter(wide_points[:, 0], wide_points[:, 1], label="breit")

ax.scatter(
    compact_model.cluster_centers_[:, 0],
    compact_model.cluster_centers_[:, 1],
    marker="X",
    s=120,
    label="Zentren kompakt",
)

ax.scatter(
    wide_model.cluster_centers_[:, 0],
    wide_model.cluster_centers_[:, 1],
    marker="P",
    s=120,
    label="Zentren breit",
)

ax.set_title("Inertia: Nähe der Punkte zu ihren Zentren")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **2.2. Gegenbeispiele erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_02_02.jpg?v=1784793854" width="250">



>* Gegenbeispiele hinterfragen scheinbar klare Cluster.
>* Projektionen, Clusterzahl und Deutung prüfen.

>* PCA-Cluster können fachlich irreführen
>* Gegenbeispiele mit Originalmerkmalen prüfen

>* PCA-Ansicht mit Originaldaten vergleichen
>* Clusterzuweisungen kritisch fachlich prüfen



In [ ]:
#@title Python-Code - Gegenbeispiele erkennen

# Dieses Beispiel zeigt Gegenbeispiele in PCA-Clustern.
# Wir vergleichen Projektion und ursprüngliche Merkmale.
# Auffällige Punkte werden sichtbar und überprüfbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir erzeugen zwei klare Gruppen mit einem versteckten Gegenbeispiel.
rng = np.random.default_rng(42)
cluster_a = rng.normal(loc=[0, 0, 0], scale=[0.45, 0.45, 0.15], size=(30, 3))
cluster_b = rng.normal(loc=[4, 4, 0], scale=[0.45, 0.45, 0.15], size=(30, 3))
counterexample = np.array([[0.2, -0.1, 5.0]])

# Die ersten zwei Merkmale dominieren die sichtbare Trennung.
features = np.vstack([cluster_a, cluster_b, counterexample])
feature_names = np.array(["Kaufhäufigkeit", "Warenkorbwert", "Retourenrisiko"])

# Eine einfache Prüfung schützt vor falschen Formen.
if features.shape != (61, 3):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Standardisierung macht Merkmale mit verschiedenen Skalen vergleichbar.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# PCA verdichtet drei Merkmale auf zwei sichtbare Achsen.
pca = PCA(n_components=2)
projected = pca.fit_transform(scaled_features)
explained = pca.explained_variance_ratio_

# K-Means gruppiert die Punkte in der zweidimensionalen Projektion.
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(projected)
centers = kmeans.cluster_centers_

# Große Restfehler zeigen Information außerhalb der Projektion.
reconstructed = pca.inverse_transform(projected)
residuals = np.linalg.norm(scaled_features - reconstructed, axis=1)
counter_index = int(np.argmax(residuals))

# Wir messen zusätzlich die Nähe zum zugewiesenen Clusterzentrum.
assigned_centers = centers[labels]
projected_distances = np.linalg.norm(projected - assigned_centers, axis=1)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Erklärte Varianz PC1+PC2: {explained.sum():.2f}")
print(f"Stärkster Gegenbeispiel-Kandidat: Punkt {counter_index}")
print(f"PCA-Restfehler dieses Punkts: {residuals[counter_index]:.2f}")
print(f"Abstand zum Projektionszentrum: {projected_distances[counter_index]:.2f}")

# Die Farbe zeigt Cluster, der Stern markiert das Gegenbeispiel.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(projected[:, 0], projected[:, 1], c=labels, cmap="viridis")
ax.scatter(projected[counter_index, 0], projected[counter_index, 1], marker="*", s=220, c="red", label="Gegenbeispiel")
ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=120, c="black", label="Zentren")

ax.set_title("Gegenbeispiel trotz plausibler PCA-Cluster")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend()
plt.show()



### **2.3. Anomalien erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_02_03.jpg?v=1784793852" width="250">



>* PCA macht ungewöhnliche Datenpunkte sichtbar
>* Anomalien sind Hinweise, nicht automatisch Fehler

>* PCA-Anomalien nicht nur visuell bewerten
>* Rekonstruktionsfehler zeigen verborgene Abweichungen

>* PCA-Anomalien sind Hinweise, keine Urteile
>* Datenvorbereitung und Fachprüfung bleiben entscheidend



In [ ]:
#@title Python-Code - Anomalien erkennen

# Wir erkennen Anomalien mit PCA-Projektionen.
# Rekonstruktionsfehler zeigt schlecht erklärte Beobachtungen.
# Ein Diagramm markiert auffällige Punkte.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten bilden zwei normale Merkmalsmuster.
rng = np.random.default_rng(42)
normal_data = rng.normal(loc=0.0, scale=0.7, size=(60, 3))

# Zwei Punkte weichen besonders in der dritten Richtung ab.
anomalies = np.array([[0.2, -0.1, 4.5], [-0.3, 0.4, -4.2]])
data = np.vstack((normal_data, anomalies))

# PCA beginnt mit Zentrierung der Merkmale.
mean_values = data.mean(axis=0)
centered_data = data - mean_values

# Die Kovarianzmatrix beschreibt gemeinsame Streuung der Merkmale.
covariance_matrix = np.cov(centered_data, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

# Größte Eigenwerte liefern die wichtigsten Hauptkomponenten.
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Wir behalten zwei Hauptkomponenten für Projektion und Rekonstruktion.
components = eigenvectors[:, :2]
projected_data = centered_data @ components
reconstructed_data = projected_data @ components.T + mean_values

# Große Rekonstruktionsfehler sind einfache Anomaliehinweise.
errors = np.linalg.norm(data - reconstructed_data, axis=1)
threshold = np.percentile(errors, 95)
is_anomaly = errors >= threshold

# Kurze Zahlen helfen beim Einordnen der PCA-Ergebnisse.
explained_ratio = eigenvalues / eigenvalues.sum()
print("Erklärte Varianz PC1 und PC2:", np.round(explained_ratio[:2], 3))
print("Schwelle für Rekonstruktionsfehler:", round(threshold, 3))
print("Markierte Punktindizes:", np.where(is_anomaly)[0].tolist())

# Die Projektion zeigt normale und auffällige Beobachtungen.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(projected_data[~is_anomaly, 0], projected_data[~is_anomaly, 1], label="normal")
ax.scatter(projected_data[is_anomaly, 0], projected_data[is_anomaly, 1], label="auffällig")

# Achsen und Titel machen die PCA-Darstellung lesbar.
ax.set_title("Anomaliehinweise in einer PCA-Projektion")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend()
plt.show()



## **3. PCA manuell**

### **3.1. Kovarianz und Eigenvektoren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_03_01.jpg?v=1784793844" width="250">



>* Kovarianz zeigt gemeinsame Merkmalsbewegungen.
>* Datenform warnt vor irreführendem K-Means.

>* Eigenvektoren zeigen wichtigste Streuungsrichtungen
>* Quer abweichende Punkte können Anomalien sein

>* PCA zeigt trügerische Clusterlösungen
>* Eigenvektoren kritisch als Diagnosewerkzeug nutzen



In [ ]:
#@title Python-Code - Kovarianz und Eigenvektoren

# Wir berechnen PCA-Richtungen aus einer Kovarianzmatrix.
# Eigenvektoren zeigen die wichtigsten Streuungsrichtungen.
# Auffällige Punkte erscheinen quer zur Hauptstreuung.

import numpy as np
import matplotlib.pyplot as plt

# Diese Punkte bilden eine schräg gestreckte Datenwolke.
data = np.array([
    [1.0, 1.2], [1.5, 1.7], [2.0, 2.1], [2.5, 2.7],
    [3.0, 3.1], [3.5, 3.6], [4.0, 4.2], [4.5, 4.6],
    [5.0, 5.1], [5.5, 5.7], [3.2, 1.2]
])

# Eine einfache Prüfung verhindert unpassende Eingabeformen.
if data.shape[1] != 2:
    raise ValueError("Dieser Unterrichtscode erwartet genau zwei Merkmale.")

# PCA beginnt mit Zentrierung und Kovarianzmatrix.
mean_point = data.mean(axis=0)
centered_data = data - mean_point
covariance_matrix = np.cov(centered_data, rowvar=False)

# Für symmetrische Kovarianzmatrizen ist eigh stabil und passend.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Die erklärte Varianz zeigt die Bedeutung jeder Richtung.
explained_ratio = eigenvalues / eigenvalues.sum()
projected_data = centered_data @ eigenvectors
side_distances = np.abs(projected_data[:, 1])

# Der größte Querabstand ist ein einfacher Anomaliehinweis.
anomaly_index = int(np.argmax(side_distances))
anomaly_point = data[anomaly_index]

print("Kovarianzmatrix gerundet:")
print(np.round(covariance_matrix, 2))
print("Erklärte Varianz PC1/PC2:", np.round(explained_ratio, 3))
print("Auffälliger Querpunkt:", np.round(anomaly_point, 2))

# Die Grafik verbindet Datenwolke, Eigenvektoren und Auffälligkeit.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(data[:, 0], data[:, 1], label="Datenpunkte")
ax.scatter(anomaly_point[0], anomaly_point[1], color="red", label="Quer auffällig")

# Die Pfeile zeigen die beiden PCA-Richtungen im Merkmalsraum.
for index, color in zip([0, 1], ["black", "orange"]):
    vector = eigenvectors[:, index] * np.sqrt(eigenvalues[index])
    ax.arrow(
        mean_point[0], mean_point[1], vector[0], vector[1],
        color=color, width=0.03, length_includes_head=True
    )

ax.set_title("Kovarianz und Eigenvektoren bei manueller PCA")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **3.2. Projektion und Varianz**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_03_02.jpg?v=1784793846" width="250">



>* PCA projiziert Daten auf strukturreiche neue Achsen
>* Projektionen zeigen Cluster und mögliche Darstellungsfehler

>* Hohe Varianz zeigt wichtige Datenunterschiede
>* Niedrige Varianz kann relevante Muster verbergen

>* PCA-Trennung zeigt Cluster, beweist sie nicht
>* Ausreißer mit Kontext und Qualitätsmaßen prüfen



In [ ]:
#@title Python-Code - Projektion und Varianz

# Wir projizieren Daten manuell auf Hauptkomponenten.
# Erklärte Varianz zeigt Informationsverlust der Projektion.
# Ausreißer werden in der PCA-Ansicht sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Kundendaten mit zwei Gruppen und einem Ausreißer.
data = np.array([
    [22, 1800, 2], [25, 2100, 3], [28, 2400, 3],
    [45, 5200, 8], [48, 5600, 9], [52, 6100, 10], [35, 9000, 1]
], dtype=float)

# Skalierung verhindert, dass Einkommen alles dominiert.
feature_means = data.mean(axis=0)
feature_stds = data.std(axis=0, ddof=1)
scaled_data = (data - feature_means) / feature_stds

# Die Kovarianzmatrix beschreibt gemeinsame Streuung der Merkmale.
covariance_matrix = np.cov(scaled_data, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
order = np.argsort(eigenvalues)[::-1]

# Größte Eigenwerte gehören zu den wichtigsten Hauptkomponenten.
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
explained_ratio = eigenvalues / eigenvalues.sum()

# Projektion bedeutet Matrixmultiplikation mit den neuen Achsen.
projected = scaled_data @ eigenvectors[:, :2]
reconstruction = projected @ eigenvectors[:, :2].T
lost_variance = 1 - explained_ratio[:2].sum()

# Einfache Anomaliehinweise entstehen durch große Projektionsabstände.
distances = np.sqrt(np.sum(projected ** 2, axis=1))
anomaly_index = int(np.argmax(distances))

print(f"Erklärte Varianz PC1: {explained_ratio[0]:.2f}")
print(f"Erklärte Varianz PC2: {explained_ratio[1]:.2f}")
print(f"Verlorene Varianz nach zwei PCs: {lost_variance:.2f}")
print(f"Auffälligster Punkt in der Projektion: Kunde {anomaly_index}")

# Die Grafik zeigt Clustertrennung und den auffälligen Punkt.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(projected[:, 0], projected[:, 1], s=80, label="Kunden")
ax.scatter(projected[anomaly_index, 0], projected[anomaly_index, 1], s=160,
           facecolors="none", edgecolors="red", label="Anomaliehinweis")

# Beschriftungen machen die neue PCA-Perspektive lesbar.
for index, point in enumerate(projected):
    ax.text(point[0] + 0.05, point[1] + 0.05, str(index))

ax.set_title("PCA-Projektion: Varianz, Cluster und Ausreißer")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend()
plt.show()



### **3.3. PCA Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_03_03.jpg?v=1784793848" width="250">



>* PCA macht wichtige Datenmuster sichtbar
>* Cluster und Ausreißer leichter beurteilen

>* PCA-Trennung kritisch auf Belastbarkeit prüfen
>* Erhaltene und verlorene Informationen beachten

>* Gegenbeispiele zeigen Grenzen einfacher Cluster.
>* Anomalien liefern prüfbare Hinweise, keine Urteile.



In [ ]:
#@title Python-Code - PCA Mini Projekt

# Dieses Mini-Projekt verbindet PCA mit Clusterprüfung.
# Wir erzeugen Gruppen und gezielte Anomalien.
# Die Grafik zeigt Struktur und Warnhinweise.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import sklearn

# Ein fester Zufallsgenerator macht das Beispiel reproduzierbar.
rng = np.random.default_rng(42)

# Drei Kundengruppen entstehen in vier Merkmalen.
centers = np.array([[2, 2, 1, 1], [7, 6, 2, 2], [3, 8, 7, 6]])
cluster_parts = []

for center in centers:
    part = rng.normal(loc=center, scale=[0.7, 0.8, 0.6, 0.7], size=(45, 4))
    cluster_parts.append(part)

# Zwei ungewöhnliche Profile dienen als einfache Anomaliehinweise.
anomalies = np.array([[9.5, 1.0, 8.5, 1.2], [0.5, 9.5, 0.8, 8.8]])
X = np.vstack(cluster_parts + [anomalies])

if X.shape != (137, 4):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Standardisierung verhindert Dominanz durch unterschiedliche Skalen.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA wird hier manuell mit Kovarianzmatrix und Eigenvektoren berechnet.
covariance_matrix = np.cov(X_scaled, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

explained_ratio = eigenvalues / eigenvalues.sum()
X_pca = X_scaled @ eigenvectors[:, :2]

# K-Means prüft, ob die sichtbare Struktur clusterbar wirkt.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

silhouette = silhouette_score(X_scaled, cluster_labels)
distances = np.linalg.norm(X_scaled - kmeans.cluster_centers_[cluster_labels], axis=1)
threshold = np.percentile(distances, 97)
anomaly_flags = distances >= threshold

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Erklärte Varianz PC1: {explained_ratio[0]:.2f}")
print(f"Erklärte Varianz PC2: {explained_ratio[1]:.2f}")
print(f"Silhouette-Wert im Originalraum: {silhouette:.2f}")
print(f"Markierte Anomaliehinweise: {int(anomaly_flags.sum())}")

# Die PCA-Ansicht zeigt Cluster, Überlappung und auffällige Punkte.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="viridis")

ax.scatter(
    X_pca[anomaly_flags, 0], X_pca[anomaly_flags, 1],
    facecolors="none", edgecolors="red", s=140, linewidths=2, label="Hinweis"
)

ax.set_title("PCA Mini Projekt: Cluster und Anomaliehinweise")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend(title="Anomalie")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Cluster und PCA**</font>


In this lecture, you learned to:
- Implementieren K-Means-Schritte mit Distanzen, Zuordnung und Zentrenaktualisierung. 
- Berechnen Hauptkomponenten, erklärte Varianz und Projektionen mit NumPy. 
- Analysieren Clusterqualität, Gegenbeispiele und einfache Anomaliehinweise. 

In the next Module (Module 9), we will go over 'scikit-learn Start'